# 1b — BATERIA overnight: qual dado ensina português ao CSM? · Colab

**Modo "rodar 1x e voltar em ~3h".** Roda vários finetunes de língua em sequência,
**compute-matched** (mesmo teto de tempo cada), salva tudo no Drive e imprime tabela.
Sem babá: PREFLIGHT (valida 1 step antes de gastar horas) + time-budget global +
checkpoints no Drive + try/except por experimento.

**Pergunta de ouro:** leitura limpa (CML-TTS, do Frederico) **vs** podcast espontâneo
(TAGARELA) **vs** mix — qual dá o menor WER pra ensinar pt ao CSM-1B?

**GPU:** A100 recomendada. **Batch fixo em 2** (valor do notebook oficial Unsloth;
batch>2 quebra o `audio_tokens_offsets` do CSM — lição de 11/jun).
**Resultado:** `Drive/TTS-ptbr-data/runs/BATERIA_results.md` + adapters em `runs/battery_*`.

In [ ]:
# ⚙️ CONFIG da bateria  (compute-matched: cada exp treina o MESMO tempo)
TIME_BUDGET_MIN = 160     # teto TOTAL da sessão (margem dos ~180 AFK)
PER_EXP_MIN     = 50      # teto de TREINO por experimento
LORA_R, LORA_ALPHA, LR = 64, 64, 5e-5
BATCH, ACCUM = 2, 16      # CSM-Unsloth: batch 2 testado; >2 quebra (efetivo 32)

BATTERY = [
    {'name': 'A1_cml',      'source': 'cml',      'hours': 30},   # leitura limpa (Frederico, CC-BY)
    {'name': 'A3_tagarela', 'source': 'tagarela', 'hours': 25},   # podcast espontâneo (stream) 🔥
    {'name': 'A2_mix',      'source': 'mix',      'hours': 40},   # leitura multi-fonte (bônus)
]

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
GH_TOKEN = userdata.get('GH_TOKEN'); os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!git clone https://{GH_TOKEN}@github.com/pedrocormann/TTS-ptbr.git /content/TTS-ptbr 2>/dev/null || (cd /content/TTS-ptbr && git pull)
%cd /content/TTS-ptbr
DRIVE = '/content/drive/MyDrive/TTS-ptbr-data'; os.makedirs(f'{DRIVE}/runs', exist_ok=True)

In [ ]:
%%capture
import re, torch
v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xf = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, '0.0.34')
!pip install sentencepiece protobuf "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xf} peft trl triton unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.52.3
!pip install --no-deps trl==0.22.2
!pip install torchcodec "datasets>=3.4.1,<4.0.0" soundfile jiwer librosa soxr faster-whisper==1.1.0

## Helpers (loaders · preprocess · treino com time-cap · eval WER · PREFLIGHT)

In [ ]:
from datasets import load_dataset, Audio, Dataset, concatenate_datasets
import numpy as np, hashlib, time, gc, json, pathlib, torch

def load_source(source, hours):
    if source == 'cml':
        ds = load_dataset('ylacombe/cml-tts', 'portuguese', split='train')
    elif source == 'mix':
        cml = load_dataset('ylacombe/cml-tts', 'portuguese', split='train')
        mls = load_dataset('facebook/multilingual_librispeech', 'portuguese', split='train')
        norm = []
        for p in [cml, mls]:
            tcol = 'text' if 'text' in p.column_names else ('transcript' if 'transcript' in p.column_names else 'sentence')
            if tcol != 'text': p = p.rename_column(tcol, 'text')
            norm.append(p.remove_columns([c for c in p.column_names if c not in ('audio','text')]))
        ds = concatenate_datasets(norm)
    elif source == 'tagarela':
        st = load_dataset('freds0/TAGARELA', split='train', streaming=True)
        rows, tot, tgt = [], 0.0, hours*3600
        for ex in st:
            a = ex['audio']; rows.append({'audio': a, 'text': ex['sentence']})
            tot += len(a['array'])/a['sampling_rate']
            if tot >= tgt: break
        ds = Dataset.from_list(rows)
    ds = ds.cast_column('audio', Audio(sampling_rate=24000)).shuffle(seed=42)
    if source != 'tagarela':
        idx, tot = [], 0.0
        for i, ex in enumerate(ds):
            tot += len(ex['audio']['array'])/24000; idx.append(i)
            if tot >= hours*3600: break
        ds = ds.select(idx)
    return ds

def build_prep(processor, max_audio):
    def spk(ex, i):
        return str(int(hashlib.md5(str(ex.get('speaker_id', i)).encode()).hexdigest(), 16) % 10)
    def prep(ex, idx):
        conv = [{'role': spk(ex, idx), 'content': [{'type':'text','text':str(ex['text']).strip()},
                                                   {'type':'audio','path':ex['audio']['array']}]}]
        o = processor.apply_chat_template(conv, tokenize=True, return_dict=True, output_labels=True,
            text_kwargs={'padding':'max_length','max_length':256,'pad_to_multiple_of':8,'padding_side':'right'},
            audio_kwargs={'sampling_rate':24000,'max_length':max_audio,'padding':'max_length'},
            common_kwargs={'return_tensors':'pt'})
        return {k: v[0] for k, v in o.items()}
    return prep

def load_csm():
    from unsloth import FastModel
    from transformers import CsmForConditionalGeneration
    return FastModel.from_pretrained(model_name='unsloth/csm-1b', max_seq_length=2048,
                                     dtype=None, auto_model=CsmForConditionalGeneration, load_in_4bit=False)

def preflight():
    """Valida 1 step de treino (~3min) ANTES de gastar horas. Aborta cedo se falhar."""
    from transformers import TrainingArguments, Trainer
    from unsloth import FastModel, is_bfloat16_supported
    print('🔍 PREFLIGHT — baixando 4 exemplos e treinando 1 step…')
    st = load_dataset('ylacombe/cml-tts', 'portuguese', split='train', streaming=True)
    rows = []
    for ex in st:
        rows.append({'audio': ex['audio'], 'text': ex['text']})
        if len(rows) >= 4: break
    raw = Dataset.from_list(rows).cast_column('audio', Audio(sampling_rate=24000))
    max_audio = min(20*24000+1, int(max(len(e['audio']['array']) for e in raw)) + 1)
    model, processor = load_csm()
    ds = raw.map(build_prep(processor, max_audio), with_indices=True, remove_columns=raw.column_names)
    model = FastModel.get_peft_model(model, r=8, lora_alpha=16, lora_dropout=0, bias='none',
        target_modules=['q_proj','k_proj','v_proj','o_proj'],
        use_gradient_checkpointing='unsloth', random_state=3407)
    tr = Trainer(model=model, train_dataset=ds, args=TrainingArguments(
        per_device_train_batch_size=BATCH, gradient_accumulation_steps=1, max_steps=1,
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=1, optim='adamw_8bit', output_dir='/tmp/preflight', report_to='none'))
    tr.train()
    del model, processor, tr, ds, raw; gc.collect(); torch.cuda.empty_cache()
    print('✅ PREFLIGHT PASSOU — 1 step OK. A bateria pode rodar segura.')

def eval_wer(model, processor, ref, out):
    import soundfile as sf, jiwer
    from faster_whisper import WhisperModel
    model.eval()
    bench = [json.loads(l) for l in open('eval/benchmark_ptbr.jsonl', encoding='utf-8') if l.strip()]
    gd = pathlib.Path(f'{out}/gen'); gd.mkdir(exist_ok=True, parents=True)
    for i, it in enumerate(bench):
        conv = [{'role':'0','content':[{'type':'text','text':str(ref['text'])},{'type':'audio','path':ref['audio']['array']}]},
                {'role':'0','content':[{'type':'text','text':it['text']}]}]
        inp = processor.apply_chat_template(conv, tokenize=True, return_dict=True)
        with torch.no_grad():
            au = model.generate(**inp.to('cuda'), max_new_tokens=375, output_audio=True,
                                do_sample=True, temperature=0.9,
                                depth_decoder_do_sample=True, depth_decoder_temperature=0.9)
        sf.write(gd / f"{it.get('id', i)}.wav", au[0].to(torch.float32).cpu().numpy(), 24000)
    asr = WhisperModel('small', device='cuda', compute_type='int8_float16')
    norm = jiwer.Compose([jiwer.ToLowerCase(), jiwer.RemovePunctuation(), jiwer.RemoveMultipleSpaces(), jiwer.Strip()])
    ws = []
    for i, it in enumerate(bench):
        segs, _ = asr.transcribe(str(gd / f"{it.get('id', i)}.wav"), language='pt')
        hyp = ' '.join(s.text.strip() for s in segs).strip()
        ws.append(jiwer.wer(norm(it['text']), norm(hyp)) if hyp else 1.0)
    del asr; gc.collect(); torch.cuda.empty_cache()
    return round(float(np.mean(ws)), 3)

def run_experiment(exp, deadline_global):
    from unsloth import FastModel, is_bfloat16_supported
    from transformers import TrainingArguments, Trainer, TrainerCallback
    name, out = exp['name'], f"{DRIVE}/runs/battery_{exp['name']}"
    os.makedirs(out, exist_ok=True); t0 = time.time()
    print(f"\n{'='*64}\n▶ {name}  ({exp['source']}, {exp['hours']}h)  {time.strftime('%H:%M')}\n{'='*64}")
    raw = load_source(exp['source'], exp['hours'])
    raw = raw.filter(lambda ex: 1.5 <= len(ex['audio']['array'])/24000 <= 20 and len(str(ex['text']).split()) >= 3)
    probe = raw.select(range(min(1500, len(raw))))
    MAX_AUDIO = min(20*24000+1, int(max(len(ex['audio']['array']) for ex in probe)) + 1)
    print(f"  {len(raw)} clipes · max_audio={MAX_AUDIO/24000:.0f}s")

    model, processor = load_csm()
    ds = raw.map(build_prep(processor, MAX_AUDIO), with_indices=True, remove_columns=raw.column_names, desc='tok')
    model = FastModel.get_peft_model(model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0, bias='none',
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        use_gradient_checkpointing='unsloth', random_state=3407)

    cap_min = min(exp.get('minutes', PER_EXP_MIN), (deadline_global - time.time())/60 - 8)
    if cap_min < 5:
        print("  ⏱ sem tempo p/ treinar — pulando")
        del model, processor, ds, raw; gc.collect(); torch.cuda.empty_cache(); return None
    print(f"  batch {BATCH}×{ACCUM} (efetivo {BATCH*ACCUM}) · cap {cap_min:.0f}min")

    class TimeCap(TrainerCallback):
        def __init__(s, m): s.dl = time.time() + m*60
        def on_step_end(s, a, st, c, **k):
            if time.time() > s.dl: c.should_training_stop = True
            return c

    tr = Trainer(model=model, train_dataset=ds, args=TrainingArguments(
        per_device_train_batch_size=BATCH, gradient_accumulation_steps=ACCUM,
        num_train_epochs=99, learning_rate=LR, lr_scheduler_type='cosine', warmup_ratio=0.03,
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=10, optim='adamw_8bit', weight_decay=0.01, seed=3407,
        output_dir=out, report_to='none', save_steps=100, save_total_limit=1),
        callbacks=[TimeCap(cap_min)])
    tr.train()
    steps = tr.state.global_step
    model.save_pretrained(f'{out}/final'); processor.save_pretrained(f'{out}/final')
    wer = eval_wer(model, processor, raw[0], out)
    r = {'name': name, 'source': exp['source'], 'hours': exp['hours'],
         'steps': steps, 'wer': wer, 'min': round((time.time()-t0)/60)}
    del model, processor, tr, ds, raw; gc.collect(); torch.cuda.empty_cache()
    print("  ✅", r)
    return r

## ▶️ Rodar — PREFLIGHT primeiro (3 min); só solta a bateria se passar

In [ ]:
import time, traceback

# 1) PREFLIGHT — se falhar, NÃO desperdiça as 3h; mostra o erro em ~3min
try:
    preflight()
    SAFE = True
except Exception as e:
    SAFE = False
    traceback.print_exc()
    print("\n❌ PREFLIGHT FALHOU — não vou rodar a bateria. Manda esse erro pro Claude.")

# 2) BATERIA — só roda se o preflight passou
results = []
if SAFE:
    deadline = time.time() + TIME_BUDGET_MIN*60
    for exp in BATTERY:
        if time.time() > deadline - 12*60:
            print(f"⏹ orçamento esgotado — não inicio {exp['name']}"); break
        try:
            r = run_experiment(exp, deadline)
            if r:
                results.append(r)
                json.dump(results, open(f'{DRIVE}/runs/BATERIA_parcial.json','w'), ensure_ascii=False, indent=1)
        except Exception as e:
            traceback.print_exc(); print(f"  ❌ {exp['name']}: {e}")

    print("\n\n" + "="*64 + "\n=== RESULTADOS DA BATERIA ===\n" + "="*64)
    md_lines = ["| exp | fonte | horas | steps | WER | min |", "|---|---|---|---|---|---|"]
    for r in sorted(results, key=lambda x: x['wer']):
        line = f"| {r['name']} | {r['source']} | {r['hours']} | {r['steps']} | {r['wer']:.1%} | {r['min']:.0f} |"
        md_lines.append(line); print(line)
    if results:
        best = min(results, key=lambda x: x['wer'])
        note = f"\n🏆 MELHOR: {best['name']} (WER {best['wer']:.1%}) → BASE-PT do Estágio B (notebook 2)\n"
        note += "Adapters: Drive/TTS-ptbr-data/runs/battery_*/final · amostras em .../gen/\n"
    else:
        note = "\n(nenhum experimento concluiu — ver erros acima)\n"
    print(note)
    open(f'{DRIVE}/runs/BATERIA_results.md','w').write("# Bateria de língua — resultados\n\n" + "\n".join(md_lines) + "\n" + note)
    print("📄 salvo: Drive/TTS-ptbr-data/runs/BATERIA_results.md")